# 逐步理解 `local_evaluator`

这份 Notebook 通过可运行的小实验解释 evaluator 的完整链路：数据加载、隐藏意图、四类场景、结构化追问、推荐清洗、命中判定、异常处理和指标计算。

> 带有 **白盒实验** 标记的单元会直接读取隐藏目标，只用于理解裁判行为，不能作为正式参赛 Agent 的实现方式。

## 0. 运行准备

建议在 VS Code 中打开本文件，并选择项目的 `.venv\Scripts\python.exe` 作为内核。如果编辑器提示该环境缺少 Notebook 内核，可在已激活的虚拟环境中运行：

```powershell
python -m pip install ipykernel
```

下面的实验代码本身只使用 Python 标准库和项目代码。请按顺序运行单元格。

In [1]:
from pathlib import Path
from pprint import pprint
from collections import Counter
import sys

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'evaluator' / 'local_evaluator.py').exists():
            return candidate
    raise RuntimeError('找不到项目根目录，请从项目目录或 notebooks 目录运行本文件。')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

from evaluator.local_evaluator import (
    ALLOWED_ATTRIBUTES, MAX_TURNS, TOP_K,
    behavior_for, catalog_index, classify_constraint, coarse_category,
    customer_reply, evaluate, initial_message, load_jsonl,
    materialize_hidden_fields, metric_summary, normalize_recommendations,
)

CATALOG_PATH = PROJECT_ROOT / 'data' / 'catalog.jsonl'
DATASET_PATH = PROJECT_ROOT / 'data' / 'public_set.jsonl'
print('项目根目录:', PROJECT_ROOT)
print('最大轮数:', MAX_TURNS, '每轮最多评分推荐数:', TOP_K)
print('允许追问的属性:', sorted(ALLOWED_ATTRIBUTES))

项目根目录: f:\repos\techjam-conversational-search
最大轮数: 10 每轮最多评分推荐数: 10
允许追问的属性: ['brand', 'budget', 'category', 'color', 'feature', 'material', 'other', 'size', 'style', 'use_case']


## 1. 加载公开会话与商品目录

Evaluator 会读取公开会话，并把 50,000 件商品建立为三个内存索引：合法 ASIN 集合、类别映射和完整商品映射。第一次运行这一格会占用一些内存。

In [2]:
samples = load_jsonl(DATASET_PATH)
catalog_ids, categories, products = catalog_index(CATALOG_PATH)

print('公开会话数:', len(samples))
print('目录商品数:', len(catalog_ids))
print('场景分布:', Counter(sample['scenario_type'] for sample in samples))

公开会话数: 200
目录商品数: 50000
场景分布: Counter({'buying': 80, 'browsing': 80, 'intent_override': 30, 'boundary': 10})


## 2. Agent 能看到什么？

公开数据为了本地评分包含 `ground_truth`，但 evaluator 不会把它传给 Agent。Agent 只能在 `reset` 收到 `user_profile`，在 `respond` 收到当前消息、轮次和 `top_k`。

In [3]:
sample = samples[0]
print('会话记录包含的键:', sorted(sample))
print('Evaluator 内部隐藏目标:', sample['ground_truth']['parent_asin'])
print('传给 Agent.reset 的 user_profile:')
pprint(sample['user_profile'])

会话记录包含的键: ['category_bucket', 'difficulty_bucket', 'ground_truth', 'sample_id', 'scenario_type', 'user_profile']
Evaluator 内部隐藏目标: B09PYB7B6Z
传给 Agent.reset 的 user_profile:
{'average_prior_rating': 5.0,
 'preference_tags': ['fit', 'comfort', 'durability'],
 'purchase_frequency': '3-4 prior purchases',
 'rating_style': 'usually positive',
 'summary': 'Prior purchases emphasize fit, comfort, durability; ratings are '
            'usually positive.'}


## 3. 隐藏意图如何从目标商品生成

当公开记录没有直接携带 `intent_card` 时，evaluator 会读取目标商品的标题、特征、详情、材质、颜色和价格，构造硬约束与软偏好。随机行为使用 `sample_id + scenario_type` 作为种子，因此相同数据的模拟行为可复现。

In [4]:
target = str(sample['ground_truth']['parent_asin'])
product = products[target]
card, behavior = materialize_hidden_fields(sample, products)

print('目标商品摘要:')
pprint({key: product.get(key) for key in ('parent_asin', 'title', 'price', 'categories', 'store')})
print('\n生成的隐藏 intent_card:')
pprint(card)
print('\n生成的场景 behavior:')
pprint(behavior)

目标商品摘要:
{'categories': ['Clothing, Shoes & Jewelry', 'Boys', 'Jewelry', 'Necklaces'],
 'parent_asin': 'B09PYB7B6Z',
 'price': 9.99,
 'store': 'QIAN0813',
 'title': 'QIAN0813 Celttic Knot Triple Moon Pentagram Pentacle Star Wicca '
          'Pendant Necklace Round Pagan Jewelry'}

生成的隐藏 intent_card:
{'hard_constraints': ['Material:alloy', 'Triple Moon Pentagram Symbol'],
 'soft_preferences': ['The Triple Moon represents the Phases of the Moon which '
                      'are linked to the three aspects of the Goddess and the '
                      'phases of the Life of Women.The Pentagram representing '
                      'the holistic r',
                      '♥ a special gift to your '
                      'wife/mom/girlfriend/daughter/grandmother/best '
                      'friend/kids on St. Valentine’s Day, Easter， Christmas '
                      'day,Birthday,Anniversary ,Independence Day,Labor '
                      'Day,Th'],
 'target_category': 'QIAN0813 Celttic 

## 4. 四种场景的第一条消息

Buying 会提前透露硬约束；Browsing 和 Boundary 从模糊需求开始；Intent Override 先给旧偏好，随后在第 3 或第 4 轮替换。

In [5]:
initial_messages = []
for scenario in ('buying', 'browsing', 'intent_override', 'boundary'):
    scenario_sample = next(item for item in samples if item['scenario_type'] == scenario)
    card, behavior = materialize_hidden_fields(scenario_sample, products)
    effective = {**scenario_sample, 'intent_card': card, 'behavior': behavior}
    target = str(scenario_sample['ground_truth']['parent_asin'])
    disclosed = set()
    message = initial_message(effective, coarse_category(categories[target]), disclosed)
    initial_messages.append({
        'scenario': scenario,
        'message': message,
        'initially_disclosed': sorted(disclosed),
    })
pprint(initial_messages)

[{'initially_disclosed': ['Material:alloy'],
  'message': "I'm looking for Jewelry Necklaces. A key requirement is: "
             'Material:alloy.',
  'scenario': 'buying'},
 {'initially_disclosed': [],
  'message': "I'm looking for Basketball Men, but I'm still exploring.",
  'scenario': 'browsing'},
 {'initially_disclosed': [],
  'message': "I'm looking for Accessories Belts. Buckle closure",
  'scenario': 'intent_override'},
 {'initially_disclosed': [],
  'message': "I'm looking for Athletic Walking, but I'm still exploring.",
  'scenario': 'boundary'}]


## 5. `ask_attribute` 才控制模拟顾客

Evaluator 不解析 `message` 的语义。真正决定顾客回复的是结构化字段 `ask_attribute`。下面对同一个 Buying 会话依次询问多个属性，观察 `disclosed` 如何避免重复透露约束。

In [6]:
buying_sample = next(item for item in samples if item['scenario_type'] == 'buying')
card, behavior = materialize_hidden_fields(buying_sample, products)
effective = {**buying_sample, 'intent_card': card, 'behavior': behavior}
target = str(buying_sample['ground_truth']['parent_asin'])
disclosed = set()
first_message = initial_message(effective, coarse_category(categories[target]), disclosed)
print('初始消息:', first_message)
print('初始已透露:', disclosed)

boundary_used = False
for attribute in (None, 'material', 'color', 'budget', 'feature', 'other'):
    reply, boundary_used = customer_reply(effective, attribute, disclosed, boundary_used)
    print(f'{attribute!r:>10} -> {reply}')
print('最终已透露:', disclosed)

初始消息: I'm looking for Jewelry Necklaces. A key requirement is: Material:alloy.
初始已透露: {'Material:alloy'}
      None -> Those options are not quite right yet. Ask me about one specific attribute.
'material' -> I don't have an additional preference for material.
   'color' -> I don't have an additional preference for color.
  'budget' -> I don't have an additional preference for budget.
 'feature' -> For that, what matters is: Triple Moon Pentagram Symbol; The Triple Moon represents the Phases of the Moon which are linked to the three aspects of the Goddess and the phases of the Life of Women.The Pentagram representing the holistic r.
   'other' -> For that, what matters is: ♥ a special gift to your wife/mom/girlfriend/daughter/grandmother/best friend/kids on St. Valentine’s Day, Easter， Christmas day,Birthday,Anniversary ,Independence Day,Labor Day,Th.
最终已透露: {'The Triple Moon represents the Phases of the Moon which are linked to the three aspects of the Goddess and the phases of the Li

## 6. Boundary 场景的特殊规则

Boundary 会话中，顾客对第一次被询问的具体属性明确表示没有偏好。这个行为只触发一次；Agent 应改问其他属性或自行判断。

In [7]:
boundary_sample = next(item for item in samples if item['scenario_type'] == 'boundary')
card, behavior = materialize_hidden_fields(boundary_sample, products)
effective = {**boundary_sample, 'intent_card': card, 'behavior': behavior}
disclosed = set()

reply1, boundary_used = customer_reply(effective, 'material', disclosed, False)
reply2, boundary_used = customer_reply(effective, 'other', disclosed, boundary_used)
print('第一次询问:', reply1)
print('换一个问题:', reply2)
print('boundary_used:', boundary_used)

第一次询问: I don't have a preference for material; please use your judgment.
换一个问题: For that, what matters is: fabric; 100% Textile.
boundary_used: True


## 7. 观察一场完整的 Intent Override 对话

这个 TraceAgent 不推荐商品，只轮流询问属性并记录收到的消息。这样可以看到 evaluator 在何时插入新的意图。

In [8]:
class TraceAgent:
    def __init__(self):
        self.calls = []

    def reset(self, session_id, user_profile):
        self.session_id = session_id

    def respond(self, session_id, user_message, turn, top_k):
        attributes = ('material', 'color', 'feature', 'other')
        ask = attributes[(turn - 1) % len(attributes)]
        self.calls.append({'turn': turn, 'user_message': user_message, 'ask_attribute': ask})
        return {
            'message': f'Tell me more about {ask}.',
            'ask_attribute': ask,
            'recommendations': [],
            'usage': {'prompt_tokens': 0, 'completion_tokens': 0},
        }

override_sample = next(item for item in samples if item['scenario_type'] == 'intent_override')
trace_agent = TraceAgent()
trace_result = evaluate(trace_agent, [override_sample], catalog_ids, categories, products)
pprint(trace_agent.calls)
print('会话结果:', trace_result['sessions'][0])

[{'ask_attribute': 'material',
  'turn': 1,
  'user_message': "I'm looking for Accessories Belts. Buckle closure"},
 {'ask_attribute': 'color',
  'turn': 2,
  'user_message': 'For that, what matters is: leather; 100% Leather.'},
 {'ask_attribute': 'feature',
  'turn': 3,
  'user_message': 'Actually, ignore my earlier preference. What I need is: '
                  'leather.'},
 {'ask_attribute': 'other',
  'turn': 4,
  'user_message': 'For that, what matters is: Imported; Buckle closure.'},
 {'ask_attribute': 'material',
  'turn': 5,
  'user_message': "I don't have an additional preference for other."},
 {'ask_attribute': 'color',
  'turn': 6,
  'user_message': "I don't have an additional preference for material."},
 {'ask_attribute': 'feature',
  'turn': 7,
  'user_message': "I don't have an additional preference for color."},
 {'ask_attribute': 'other',
  'turn': 8,
  'user_message': "I don't have an additional preference for feature."},
 {'ask_attribute': 'material',
  'turn': 9,
  

## 8. 推荐列表如何被标准化

Evaluator 保留原顺序，但会删除空值、重复项和目录中不存在的 ASIN，并在得到 10 个合法唯一值后停止。虽然标准化函数也接受裸字符串，正式 Agent 应遵守契约返回 `{"parent_asin": ...}`。

In [9]:
valid = sorted(catalog_ids)[:12]
raw_recommendations = [
    {'parent_asin': valid[0]},
    {'parent_asin': valid[0]},       # 重复
    {'parent_asin': 'NOT_IN_CATALOG'},
    '',
    *({'parent_asin': asin} for asin in valid[1:]),
]
normalized = normalize_recommendations(raw_recommendations, catalog_ids)
print('输入数量:', len(raw_recommendations))
print('评分列表:', normalized)
print('评分数量:', len(normalized))

输入数量: 15
评分列表: ['0800732022', '5000000056', '5000000064', '7750000348', '9479290707', '9999666671', 'B00007FFM1', 'B00008ID1A', 'B00008NMKE', 'B00009EIVN']
评分数量: 10


## 9. 白盒实验：精确命中、排名、轮次与 Token

下面的 Agent 直接知道隐藏目标，只用于验证评分机制。我们让它在第 2 轮把目标放在第 3 名；预期 `first_hit_turn=2`、`best_rank=3`、`reciprocal_rank=1/3`。

In [10]:
class WhiteBoxAgent:
    def __init__(self, target, decoys, hit_turn=1, rank=1):
        self.target = target
        self.decoys = [asin for asin in decoys if asin != target]
        self.hit_turn = hit_turn
        self.rank = rank

    def reset(self, session_id, user_profile):
        pass

    def respond(self, session_id, user_message, turn, top_k):
        recommendations = self.decoys[:top_k]
        if turn >= self.hit_turn:
            recommendations = self.decoys[:self.rank - 1] + [self.target]
            recommendations += self.decoys[self.rank - 1:top_k - 1]
        return {
            'message': 'White-box scoring experiment.',
            'ask_attribute': 'other',
            'recommendations': [{'parent_asin': asin} for asin in recommendations],
            'usage': {'prompt_tokens': 10, 'completion_tokens': 2},
        }

buying_sample = next(item for item in samples if item['scenario_type'] == 'buying')
target = str(buying_sample['ground_truth']['parent_asin'])
decoys = [asin for asin in sorted(catalog_ids) if asin != target][:20]
white_box = WhiteBoxAgent(target, decoys, hit_turn=2, rank=3)
white_result = evaluate(white_box, [buying_sample], catalog_ids, categories, products)
pprint(white_result['sessions'][0])
print('累计 token:', white_result['reported_token_usage'])

{'best_rank': 3,
 'first_hit_turn': 2,
 'hit': True,
 'reciprocal_rank': 0.3333333333333333,
 'sample_id': 'public_0001',
 'scenario_type': 'buying'}
累计 token: {'prompt_tokens': 20, 'completion_tokens': 4, 'total_tokens': 24}


## 10. 白盒实验：Intent Override 的命中门控

即使目标从第 1 轮开始就在推荐列表中，Intent Override 会话也必须等新意图正式出现后才能命中。

In [11]:
override_sample = next(item for item in samples if item['scenario_type'] == 'intent_override')
target = str(override_sample['ground_truth']['parent_asin'])
card, behavior = materialize_hidden_fields(override_sample, products)
expected_override_turn = behavior['override']['turn']
decoys = [asin for asin in sorted(catalog_ids) if asin != target][:20]
always_target = WhiteBoxAgent(target, decoys, hit_turn=1, rank=1)
override_result = evaluate(always_target, [override_sample], catalog_ids, categories, products)
print('新意图出现轮次:', expected_override_turn)
print('Evaluator 记录命中轮次:', override_result['sessions'][0]['first_hit_turn'])
pprint(override_result['sessions'][0])

新意图出现轮次: 3
Evaluator 记录命中轮次: 3
{'best_rank': 1,
 'first_hit_turn': 3,
 'hit': True,
 'reciprocal_rank': 1.0,
 'sample_id': 'public_0002',
 'scenario_type': 'intent_override'}


## 11. 异常为何可能表现为低分而不是崩溃

Evaluator 捕获 `respond()` 的异常，并把该轮替换为空响应。下面的 Agent 每轮都报错，但整个评估仍会结束；结果是未命中，异常次数为 10。开发 Agent 时应自行记录异常。

In [12]:
class ExplodingAgent:
    def __init__(self):
        self.exception_count = 0

    def reset(self, session_id, user_profile):
        pass

    def respond(self, session_id, user_message, turn, top_k):
        self.exception_count += 1
        raise RuntimeError('故意制造的错误')

exploding = ExplodingAgent()
error_result = evaluate(exploding, [buying_sample], catalog_ids, categories, products)
print('respond 异常次数:', exploding.exception_count)
pprint(error_result['sessions'][0])

respond 异常次数: 10
{'best_rank': None,
 'first_hit_turn': None,
 'hit': False,
 'reciprocal_rank': 0.0,
 'sample_id': 'public_0001',
 'scenario_type': 'buying'}


## 12. 用三个合成会话验证指标公式

这里构造：第 1 轮第 1 名命中、第 3 轮第 4 名命中、一个未命中。未命中的 MTTC 按第 11 轮计算，MRR 按 0 计算。

In [13]:
synthetic_sessions = [
    {'hit': True,  'first_hit_turn': 1,    'best_rank': 1,    'reciprocal_rank': 1.0},
    {'hit': True,  'first_hit_turn': 3,    'best_rank': 4,    'reciprocal_rank': 0.25},
    {'hit': False, 'first_hit_turn': None, 'best_rank': None, 'reciprocal_rank': 0.0},
]
summary = metric_summary(synthetic_sessions)
efficiency = max(0.0, min(1.0, (11.0 - summary['mttc']) / 10.0))
technical_score = (
    0.50 * summary['hit_rate_at_10']
    + 0.30 * summary['mrr']
    + 0.20 * efficiency
)
pprint(summary)
print('Efficiency:', round(efficiency, 6))
print('Technical Score:', round(technical_score, 6))

{'hit_rate_at_10': 0.666667, 'mrr': 0.416667, 'mttc': 5.0, 'sample_count': 3}
Efficiency: 0.6
Technical Score: 0.578334


## 13. 运行完整 BM25 基线

最后使用真正的 `starter.Agent` 跑完 200 个公开会话。它会在内存中建立 SQLite FTS5 索引；结果应与 `docs/baseline_results.json` 一致。这里不写入 `results.json`，只打印汇总。

In [14]:
from starter.agent import Agent

baseline_agent = Agent(CATALOG_PATH)
try:
    baseline_result = evaluate(baseline_agent, samples, catalog_ids, categories, products)
finally:
    baseline_agent.connection.close()

pprint({key: value for key, value in baseline_result.items() if key != 'sessions'})
print('命中轮次分布:', Counter(
    row['first_hit_turn'] for row in baseline_result['sessions'] if row['hit']
))

{'efficiency': 0.119,
 'hit_rate_at_10': 0.125,
 'mrr': 0.068034,
 'mttc': 9.81,
 'recommended_technical_score': 0.10671,
 'reported_token_usage': {'completion_tokens': 0,
                          'prompt_tokens': 0,
                          'total_tokens': 0},
 'sample_count': 200,
 'scenario_metrics': {'boundary': {'hit_rate_at_10': 0.0,
                                   'mrr': 0.0,
                                   'mttc': 11.0,
                                   'sample_count': 10},
                      'browsing': {'hit_rate_at_10': 0.025,
                                   'mrr': 0.004514,
                                   'mttc': 10.75,
                                   'sample_count': 80},
                      'buying': {'hit_rate_at_10': 0.2375,
                                 'mrr': 0.126508,
                                 'mttc': 8.625,
                                 'sample_count': 80},
                      'intent_override': {'hit_rate_at_10': 0.133333,
     

## 14. 建议继续做的实验

1. 修改 TraceAgent，让它记录并累计顾客已经透露的约束。
2. 分别固定 `ask_attribute` 为 `material`、`feature` 和 `other`，比较得到的信息。
3. 让 Agent 每轮避免重复推荐，并观察 MTTC 和 Hit Rate 的变化。
4. 分别只评估 Buying、Browsing、Intent Override 和 Boundary，针对场景调整策略。
5. 在 `starter/agent.py` 中加入会话状态后，重新运行最后一个单元，与基线比较。

最重要的边界：自然语言 `message` 不驱动模拟器；`ask_attribute` 决定顾客回复；精确的 `parent_asin`、排序和命中轮次决定技术得分。